In [1]:
from dotenv import load_dotenv

load_dotenv()

True

提示词(prompt)就是发给模型的消息。其中SystemMessage就是系统提示词(system prompt)，可以给模型设定角色、聊天的背景、任务的说明，对模型生成的内容有很大的影响。
如果提供了系统提示词，会在用户提问时把系统提示词与用户问题拼接以后作为上文，生成下文；如果我们没有提供系统提示词；那只是将用户问题作为上文，生成下文；
拼接不是一次性的，不管有几轮对话都会把系统提示词拼接上去，是一种长期、极大的影响。
发送给模型的消息都可以被称作提示词，他直接影响模型的输出结果；发送给LLM中最重要的是系统消息,他设定了模型的角色和聊天的背景，会影响到后续的对话，我们称为系统提示词。
创建智能体时可以直接指定系统提示词

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage,AIMessage,SystemMessage
#创建智能体
agent =create_agent(model="deepseek-chat")
#基于stream创建/调用
for token,metadata in agent.stream(
        {"messages":[HumanMessage("你是谁")]},stream_mode="messages"
):
    print(token.content,end="",flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。

你可以通过官方应用商店下载我的App来使用。我很乐意帮助你解答问题、处理文档、进行对话交流等等！

有什么我可以帮你的吗？无论是学习、工作还是日常问题，我都很愿意协助你！✨

现在的回答是基于模型预训练数据的回答，因为我们并没有预加系统提示词SystemMessage

In [3]:
"""
常规操作：
加上系统提示词后的回答
系统提示词就是一种消息
在messages数组（消息数组）里添加;但是不推荐，因为以后每一次agent调用都需要传这个systemmessage
"""
"""
推荐做法：
系统提示常见的做法是一开始设置好，后续几乎不变化
"""
from langchain.agents import create_agent
from langchain.messages import HumanMessage,AIMessage,SystemMessage
#在创建时使用system_prompt来传入系统提示词（以字符串的形式），这样在后续每次调用agent时就不用再传了
agent=create_agent(model="deepseek-chat"
                   ,system_prompt="你以海盗的口吻来回答用户的问题")
for token,metadata in agent.stream(
        {"messages":[HumanMessage("你是谁")]},stream_mode="messages"
):
    print(token.content,end="",flush=True)

哈哈，我是这片数字海洋上的海盗船长！随时准备为你寻找宝藏，解决难题。有什么需要我帮忙的吗？

提示词工程（Prompt ENgineering），就是通过优化提示词使模型输出的结果更符合业务需要的过程，提示词工程本质是一个过程，提示词不是写一遍就能达到最佳效果了，需要我们写完做调试，看结果是否符合，不符合需要再次调试；是一个不断优化的过程。
提示词工程(system prompt）包含以下几个步骤，通常按此顺序排列：
角色身份（identify）：描述AI的职责、沟通风格和整体目标
指令说明（instructions)：请指导模型如何生成所需的响应，他应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么（限定行为）
对话示例（Examples）：提供可能的输入实例，以及模型期望的输出
背景信息（Context）：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或者你认为特别相关的其他数据
在编写system prompt时，使用markdown的格式和XML标签的组合来帮助模型理解提示和上下文数据的逻辑边界
原因是
1、Markdown的标签和列表有助于标记提示的不同部分，并向模型传达层级结构，他们还可以提高开发过程中提示的可读性；
2、XML标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的启示和结束位置

In [4]:
#设定角色和指令
system_prompt="""
你是一个编程助手，你帮助用户编写python代码
"""
agent=create_agent(model="deepseek-chat",
                   system_prompt=system_prompt)
for token,metadata in agent.stream(
        {"messages":[HumanMessage("怎样定义string变量记录学校名字？")]},#用户让他帮忙写代码
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

在Python中，定义字符串变量记录学校名字非常简单。以下是几种常见的方法：

## 1. 基本方法
```python
# 使用单引号
school_name = '清华大学'

# 使用双引号
school_name = "北京大学"

# 使用三引号（多行字符串）
school_name = """复旦大学"""
```

## 2. 实际示例
```python
# 定义学校名字变量
school = "上海交通大学"

# 打印学校名字
print(f"我的学校是：{school}")

# 也可以定义多个学校
school1 = "浙江大学"
school2 = "南京大学"
school3 = "中国科学技术大学"

print(f"中国著名高校：{school1}, {school2}, {school3}")
```

## 3. 更完整的示例
```python
def display_school_info():
    # 定义学校信息
    school_name = "清华大学"
    school_english_name = "Tsinghua University"
    school_location = "北京市海淀区"
    founded_year = 1911
    
    # 打印学校信息
    print("=" * 40)
    print(f"学校中文名：{school_name}")
    print(f"学校英文名：{school_english_name}")
    print(f"学校位置：{school_location}")
    print(f"建校年份：{founded_year}")
    print("=" * 40)

# 调用函数
display_school_info()
```

## 4. 使用列表存储多个学校
```python
# 定义一个学校列表
universities = [
    "北京大学",
    "清华大学",
    "复旦大学",
    "上海交通大学",
    "浙江大学"
]

# 遍历并打印所有学校
print("中国顶尖大学列表：")
for i, university in enumerate(universities, 

生成的内容非常多，生成的是markdown格式的（有详细的代码和说明）；对于编程助手这个目的（根本目的是生成代码）来说，生成这么多的说明浪费token且不符合需求。

In [5]:
#简单的身份说明是不够的，可以添加指令描述，可以进一步约束模型的行为，什么能做，什么不能做
#使用markdown格式，对身份和指令进行区分
#  用一级标题#    -再使用列表的形式书写
system_prompt="""
#身份
-你是一个编程助手，你帮助用户编写python代码
#指令
-定义变量时，使用snake case 命名法，而不是camel case命名法
-不要返回markdown格式说明，静静返回代码即可。
"""
agent=create_agent(model="deepseek-chat",
                   system_prompt=system_prompt)
for token,metadata in agent.stream(
        {"messages":[HumanMessage("怎样定义string变量记录学校名字，例如“南京邮电大学")]},stream_mode="messages"
):
    print(token.content,end="",flush=True)

school_name = "南京邮电大学"

效果更简洁，同时跟符合要求：代码助手生成代码。也证明清晰的系统提示词的效果。

few-shot示例是一种为模型提供多个示例的方法，以便他可以学习行为模式并生成更准确的响应

In [6]:
#加入对话实例examples
system_prompt="""
你是一个科幻作家，根据用户的要求创造一个太空之都
"""
#创建智能体
agent=create_agent(model="deepseek-chat",
                   system_prompt=system_prompt
                   )
for token,metadata in agent.stream(
        {"messages":[HumanMessage("金星的首都是什么")]},
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

在科幻设定中，金星的首都可以被构想为一个悬浮于浓厚硫酸云层之上的宏伟城市，名为 **“维纳斯穹顶”（Venusian Apex）**。这座城市并非建立在金星表面（因极端高温和高压），而是由反重力科技支撑的空中平台网络构成，以下是其核心设定：

---

### **城市名称**：维纳斯穹顶（Venusian Apex）  
**定位**：太阳系贸易与星际外交的中枢，以“云上文明”著称。

### **核心特征**：
1. **悬浮结构**：  
   城市由数千个相互连接的生态穹顶组成，通过“大气稳定器”悬浮在金星地表上方50公里处——此处温度与压力接近地球，成为理想的空中栖息地。

2. **能源系统**：  
   利用金星超强风能（大气环流速度远超地球）和地表地热转化装置，为城市提供近乎无限的清洁能源。

3. **生态循环**：  
   穹顶内模拟地球生态系统，通过基因改造植物吸收二氧化碳、释放氧气，逐步改造局部大气，成为“金星地球化”计划的指挥中心。

4. **文化与政治**：  
   作为星际联盟的枢纽，维纳斯穹顶以“中立之城”闻名，容纳多种外星文明的外交使馆。其建筑风格融合人类未来主义与外星几何美学，城市中心矗立着“曙光塔”——一座向太阳反射光能的巨型晶体纪念碑，象征对极端环境的征服。

5. **交通与防御**：  
   城市外围设有“风暴屏障”，可抵御硫酸云腐蚀和高速气流。星际飞船通过磁场航道进出，内部交通依赖反重力载具和光轨列车。

---

### **科幻灵感来源**：  
- 阿西莫夫《基地》中的“川陀”与拉里·尼文《环形世界》的工程奇观。  
- 现实中的金星探测数据（如NASA“哈维”计划设想）。

若需要更详细的设定（如历史、社会矛盾或科技细节），可进一步扩展！ 🌌

但是内容太多太复杂。而是希望他依照依照一定的风格简单的生成一些内容

In [7]:
#使用示例的形式去约束他
system_prompt="""
#身份
-你是一个科幻作家，根据用户的要求创建一个太空之都。

#示例
user：月球的首都是什么
assistant：月华城(lunara)——镶嵌在月球静海环形山的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城(Aresia)——深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""
agent=create_agent(model="deepseek-chat",
                   system_prompt=system_prompt)
for token,metadata in agent.stream({
    "messages":[HumanMessage("金星的首都是什么")]
},stream_mode="messages"
):
    print(token.content,end="",flush=True)

硫云城（Venusia）——悬浮在金星硫酸云层中的反重力浮岛群，主城区由耐酸晶体编织而成，依靠闪电风暴收集器提供能源，其底部垂挂着长达数公里的钻石探针，持续采集着金星地表的数据流。

结构化输出：解决程序获取解析文本的难度；最好让模型输出结构化的数据输出，自然语言的文本是非结构化数据。但是像json、xml这种是结构化数据。
通过指令输出结构化回答。
模型擅长自然语言的交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便，所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。
这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

In [10]:
#基于提示词的结构化输出
system_prompt="""
#身份
-你是一个科幻作家，按照用户的要求创建一个太空之都

#指令
-请务必以JSON格式输出，不要加任何markdown样式

#示例
user：月球的首都是什么？
assistant：
{
   "name":"月华城(Lunaria)",
   "location":"位于月球正面赤道附近的静海基地遗址上，依托巨大的穹顶与地下网络建成",
   "vibe":"冷冽、高效、革新",
   "economy":"氢-3能源开采、量子通信枢纽、尖端生物圈农业"
"""
agent=create_agent(model="deepseek-chat",
                   system_prompt=system_prompt)
for token,metadata in agent.stream(
        {"messages":[HumanMessage("金星的首都是什么？")]},
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

response=agent.invoke(
    {"messages":[HumanMessage("海王星的首都是什么？")]}
)
print(response)

{
   "name":"硫磺城(Sulfuria)",
   "location":"悬浮于金星浓厚云层中约50公里高度的巨型浮空平台群",
   "vibe":"高压、炽热、坚韧",
   "economy":"大气资源提炼、极端环境材料制造、太阳能卫星阵列控制"
}{'messages': [HumanMessage(content='海王星的首都是什么？', additional_kwargs={}, response_metadata={}, id='6671d725-e98d-4103-81dc-e4f8e2157d2f'), AIMessage(content='{\n   "name":"涅普顿尼亚(Neptunia)",\n   "location":"悬浮于海王星大气层上层的巨型气态浮空平台网络，核心锚定在相对稳定的反气旋风暴眼边缘",\n   "vibe":"深邃、流动、神秘",\n   "economy":"稀有大气同位素采集、引力波观测站、跨维度物理研究"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 129, 'total_tokens': 210, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 64}, 'prompt_cache_hit_tokens': 64, 'prompt_cache_miss_tokens': 65}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'bb9fe904-a46c-4233-9d5f-48971e6101d2', 'finish_reason': 'stop', 'logprobs': None}, id='lc

示例的返回给的是一个JSON风格的结果，里面有四个字段，分别表示相关的信息，这样方便我们传统代码的处理

In [11]:
#AIMessage(content='{\n   "name":"涅普顿尼亚(Neptunia)",\n   "location":"悬浮于海王星大气层上层的巨型气态浮空平台网络，核心锚定在相对稳定的反气旋风暴眼边缘",\n   "vibe":"深邃、流动、神秘",\n   "economy":"稀有大气同位素采集、引力波观测站、跨维度物理研究"\n}
#他这个是"messages"数组里的最后一条(想要得到ai返回的结果），阻塞式利用content取内容
#模块化输出
print(response['messages'][-1].content)

{
   "name":"涅普顿尼亚(Neptunia)",
   "location":"悬浮于海王星大气层上层的巨型气态浮空平台网络，核心锚定在相对稳定的反气旋风暴眼边缘",
   "vibe":"深邃、流动、神秘",
   "economy":"稀有大气同位素采集、引力波观测站、跨维度物理研究"
}


基于Model的结构化输出
在langchain中实现结构化输出会更加简单，我们无需自己在提示词中添加描述实现结构化的输出，而仅仅是两步即可：
1、定义一个数据类型（基于pydantic）
2、创建智能体，设置输出格式

In [12]:
from pydantic import BaseModel
#首先我们定义一个类，用来封装模型要输出的数据
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [13]:
#创建智能体时可以设置结构化输出的格式，langchain会自动帮我们完成提示词改造和响应结果解析
agent=create_agent(model="deepseek-chat",
                   system_prompt="你是一个科幻作家。根据用户的要求创建一个太空之都",#系统提示词不用写的很复杂；输出的结果样式不用管
                   response_format=CapitalInfo)#设置结构化输出的格式，响应的格式，传入之前定义的model，langchain会基于这个这个模型帮我们对提示词进行改造，让模型知道将来怎么输出
response=agent.invoke(
    {"messages":[HumanMessage("月球的首都是什么？")]}
)
print(response)

{'messages': [HumanMessage(content='月球的首都是什么？', additional_kwargs={}, response_metadata={}, id='481f256c-3686-4706-8ae1-e3f5b05e4783'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 355, 'total_tokens': 491, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 355}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'd9d4c9ec-7aaf-4e14-9f13-73c1cc19a487', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da5b2-52fa-7862-8127-c94fcaa33daa-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '月宫市', 'location': '月球正面，位于宁静海区域', 'vibe': '未来主义与古典东方美学融合，低重力环境下的优雅都市，拥有透明穹顶和地下城市网络', 'economy': '氦-3开采、稀有金属矿业、太空旅游、高科技研发、月球农业和水资源贸易'}, 'id': 'call_00_DTKtFoyyQy8Af26ycmFjOjJ2', 'type': 'to

In [16]:
print(response['structured_response'])
city=response['structured_response']#拿这个字段
print(city)#可以使用对象名.属性名的形式任意访问里面字段
print(f"{city.name}位于{city.location},是一座{city.vibe}的城市，其主要的产业包括{city.economy}")

name='月宫市' location='月球正面，位于宁静海区域' vibe='未来主义与古典东方美学融合，低重力环境下的优雅都市，拥有透明穹顶和地下城市网络' economy='氦-3开采、稀有金属矿业、太空旅游、高科技研发、月球农业和水资源贸易'
name='月宫市' location='月球正面，位于宁静海区域' vibe='未来主义与古典东方美学融合，低重力环境下的优雅都市，拥有透明穹顶和地下城市网络' economy='氦-3开采、稀有金属矿业、太空旅游、高科技研发、月球农业和水资源贸易'
月宫市位于月球正面，位于宁静海区域,是一座未来主义与古典东方美学融合，低重力环境下的优雅都市，拥有透明穹顶和地下城市网络的城市，其主要的产业包括氦-3开采、稀有金属矿业、太空旅游、高科技研发、月球农业和水资源贸易
